<a href="https://colab.research.google.com/github/meghanakr2023/AI-Interview-Assistant/blob/main/Docu_chat_application_RAG_agents.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Load

In [ ]:
!pip install -qU langchain-community pypdf

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from pprint import pprint

In [ ]:
file_path = "/content/Attention is all you need.pdf"
loader = PyPDFLoader(file_path)
doc = loader.load()

In [ ]:
pprint(doc[0].metadata)

In [ ]:
pprint(doc[0].page_content)

Split

In [ ]:
!pip install -qU langchain-text-splitters

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
  chunk_size=1000,
  chunk_overlap=200,
)
all_splits = text_splitter.split_documents(doc)

In [ ]:
print(f"Paper split into {len(all_splits)} sub-documents.")
pprint(f"Metadata: {all_splits[0].metadata}")

Embed and store

In [ ]:
!pip install -U langchain langchain-community langchain-chroma chromadb opentelemetry-api==1.24.0 opentelemetry-sdk==1.24.0

In [ ]:
! pip install -qU langchain langchain-huggingface sentence_transformers

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
  model_name = "sentence-transformers/all-mpnet-base-v2"
)

In [ ]:
!pip install -U langchain langchain-community langchain-chroma chromadb

In [ ]:
!pip install -qU langchain-chroma

In [ ]:
from langchain_chroma import Chroma

In [ ]:
vector_store = Chroma(
  collection_name="Gen_AI_research_collection",
  embedding_function=embedding_model,
  persist_directory="./langchain_chroma_db"
)
document_ids = vector_store.add_documents(documents=all_splits)

In [ ]:
sample = vector_store.get(limit=1, include=["embeddings", "documents"])

In [ ]:
print(sample)

In [ ]:
print(document_ids[:3])

Retreival

In [ ]:
from langchain.tools import tool

In [ ]:
@tool
def retrieve_from_pdf(query: str, k: int = 2)-> str:
  """
  Retrieve information from the Attention Is All You Need research paper.
  """
  retrieved_docs = vector_store.similarity_search(query, k=k)

  docs_content = ""
  for doc in retrieved_docs:
    docs_content += f"Source: {doc.metadata}\n"
    docs_content += f"Content: {doc.page_content}\n\n"

  return docs_content, retrieved_docs

In [ ]:
result = retrieve_from_pdf("What is the use of decoders in Transformers?")

In [ ]:
pprint(result)

In [ ]:
!pip install langchain-tavily

In [ ]:
from langchain_tavily import TavilySearch

In [ ]:
tavily_api_key = userdata.get('TEVILY_API_KEY')

web_search_tool = TavilySearch(
    max_results=5,
    search_depth="advanced",
    tavily_api_key=tavily_api_key,
)

Generation

In [ ]:
!pip install -U langchain-google-genai

In [ ]:
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from google.colab import userdata

In [ ]:
api_key = userdata.get('GEMINI_API_KEY')
model = init_chat_model(
   "google_genai:gemini-2.5-flash",
   api_key=api_key,
)

In [ ]:
system_prompt = """You are a helpful research assistant with access to two tools:

1. retrieve_from_pdf: Use this to find information from the
   "Attention Is All You Need" research paper

2. TavilySearch: Use this to find current information
   not in the paper (recent events, updates, etc.)

Strategy:
- For questions about the paper content → use retrieve_from_pdf
- For questions about recent events or topics not in the paper → use TavilySearch
- DON'T make up things
"""

In [ ]:
agent = create_agent(
    model=model,
    tools=[retrieve_from_pdf, web_search_tool],
    system_prompt=system_prompt,
    debug = True
)

In [ ]:
def docu_chat(user_query):
  context, source_docs = retrieve_context(user_query, k=2)
  system_message = f"""You are a helpful chatbot.
                     Use only the following pieces of context to answer the
                     question. Don't makeup any new information: {context} """

  messages = [
    {"role": "system", "content": system_message},
    {"role": "user", "content": user_query}
  ]
  response = model.invoke(messages)
  return {
      "answer": response.content,
      "source_documents": source_docs,
      "context_used": context
  }

In [ ]:
user_query = "Compare the attention mechanism from the paper with recent improvements like Flash Attention, and tell me which approach would be better for my college project"

In [ ]:
response = agent.invoke({
    "messages": [{"role": "user", "content": user_query}]
})

In [ ]:
print(response["messages"][-1].content[0]["text"])

In [ ]:
result = docu_chat(user_query)

In [ ]:
print(result)

In [ ]:
print(result["answer"])